# ML trials (train Jan–May, test June): bottom-up coin-in models
**Goal:** predict each machine's daily coin-in, then add the predictions up by game × zone, zone and floor. Compare models and feature sets against two baselines on a time-based holdout: **train January–May, test June** (June is the hottest, softest month, so this is the toughest test).

- **Target:** log of coin-in per available hour (machine counts and downtime don't distort demand). Expected win = predicted coin-in × (1 − RTP), so hold is not modelled separately (daily hold is mostly luck).
- **Features:** things known in advance (weekday, event class, game, zone/bank, game traits), plus **temperature** from NOAA (`real_sources/weather_daily`).
- **Temperature caveat:** actual weather is not known weeks ahead. Month-ahead trials that use it are labelled **(oracle)**: they assume a perfect weather forecast and show the *most* temperature could help. In the next-day model, temperature is legitimate (tomorrow's forecast is available).
- **Needs:** `pip install scikit-learn plotly nbformat`

## 1. Setup

In [53]:
# ML trials: predict each machine's daily coin-in, then add up by game, zone and floor
import decimal, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path(os.getcwd())
while not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"

TRAIN_END = "2025-06-01"      # train on everything before this date, test on everything after
TEST_LABEL = "June"

def load(relative_path):
    """Read a parquet file and turn exact-decimal money columns into normal numbers."""
    df = pd.read_parquet(DATA / relative_path)
    for c in df.columns:
        if df[c].dtype == object and df[c].dropna().map(type).eq(decimal.Decimal).any():
            df[c] = df[c].astype(float)
    return df

In [54]:
# Plotly setup: one dark "Gilded Ace" style for every chart. save() shows each chart here AND writes it
# to its own HTML file next to index.html, so the dashboard can show it in an <iframe>.
import datetime
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

BG, WHITE = "#11244D", "#F2F2F2"
CYAN, GOLD, RED, GREEN, PURPLE, ORANGE = "#00D8FF", "#EAA244", "#E4572E", "#3BB273", "#B07CFF", "#FF9F1C"
GREY, FAINT, GRID = "#8A94A6", "rgba(255,255,255,0.22)", "rgba(255,255,255,0.08)"
TIER_COLORS = {"Base": GREY, "Gold": GOLD, "Platinum": PURPLE}

pio.templates["gilded"] = go.layout.Template(layout=dict(
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family="Inter, sans-serif", color=WHITE, size=13),
    colorway=[CYAN, GOLD, RED, GREEN, PURPLE, ORANGE, "#FF8FB1", "#7FD1B9", GREY],
    xaxis=dict(gridcolor=GRID, zerolinecolor=FAINT, linecolor=FAINT, automargin=True),
    yaxis=dict(gridcolor=GRID, zerolinecolor=FAINT, linecolor=FAINT, automargin=True),
    legend=dict(bgcolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=30, t=70, b=60),
))
pio.templates.default = "plotly_dark+gilded"

CHART_DIR = ROOT   # repo root (found in the setup cell), where index.html lives
PAGE_STYLE = f"<style>html,body{{margin:0;height:100%;background:{BG};}}</style>"

def save(fig, name, height=450, slider=None):
    """Show the chart in the notebook, then write CHART_DIR/<name>.html sized to fill its iframe.
    slider: zoom slider under the x-axis. None = automatic: every single-panel chart with a date axis gets one."""
    fig.update_layout(height=height)
    if slider is None:
        single_x = not any(k.startswith("xaxis") and k != "xaxis" for k in fig.layout.to_plotly_json())
        first_x = next((t.x for t in fig.data if getattr(t, "x", None) is not None and len(t.x)), None)
        first_val = next((v for v in first_x if v is not None), None) if first_x is not None else None
        if isinstance(first_val, str):                       # dates stored as text, e.g. "2025-01-01"
            try:
                first_val = pd.Timestamp(first_val) if len(first_val) >= 8 and first_val[:4].isdigit() else None
            except ValueError:
                first_val = None
        slider = single_x and isinstance(first_val, (datetime.date, np.datetime64, pd.Timestamp))
    print(f"zoom slider: {'yes' if slider else 'no'}")
    # Bars with labels outside them: leave headroom past the longest bar so labels like "+23%" never get cut off
    for orient, axis in [("h", "xaxis"), ("v", "yaxis")]:
        bars = [t for t in fig.data if t.type == "bar" and t.textposition == "outside"
                and (t.orientation or "v") == orient]
        if bars and fig.layout[axis].range is None:
            vals = np.concatenate([np.asarray(t.x if orient == "h" else t.y, dtype=float) for t in bars])
            lo, hi = np.nanmin(vals), np.nanmax(vals)
            fig.update_layout({axis: dict(range=[min(0, lo) * 1.18, max(0, hi) * 1.18])})
            for t in bars:
                t.cliponaxis = False
    if slider:
        # A thick, gold-framed zoom strip so it's obvious it can be dragged
        fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.13, bgcolor="rgba(234,162,68,0.10)",
                                          bordercolor=GOLD, borderwidth=2))
        legend_at_bottom = (fig.layout.legend.orientation == "h" and
                            (fig.layout.legend.y if fig.layout.legend.y is not None else 1) < 0)
        if legend_at_bottom:                          # a bottom legend would collide with the slider: move it on top
            fig.update_layout(legend=dict(x=0, y=1.02, yanchor="bottom"))
            if fig.layout.title.text:                 # and lift the title clear of it
                fig.update_layout(title=dict(y=0.98, yref="container", yanchor="top"), margin_t=110)
        fig.update_layout(margin_b=60)
    fig.show()
    page = go.Figure(fig)
    page.update_layout(height=None, width=None, autosize=True)
    # Dashboard copy: no inner title (the card above the chart already has one, and long titles get cut off)
    legend_on_top = page.layout.legend.y is not None and page.layout.legend.y >= 1
    page.update_layout(title=dict(text=""), margin_t=60 if legend_on_top else 40)
    html = page.to_html(include_plotlyjs="cdn", full_html=True, default_width="100%", default_height="100%",
                        config={"displayModeBar": False, "responsive": True})
    with open(f"{CHART_DIR}/{name}.html", "w", encoding="utf-8") as f:
        f.write(html.replace("<head>", "<head>" + PAGE_STYLE, 1))
    print(f"saved {name}.html")

def date_line(fig, date, label="", color=RED):
    """Dashed vertical line on a date axis (add_vline's built-in label breaks on dates)."""
    fig.add_shape(type="line", x0=date, x1=date, y0=0, y1=1, yref="paper", line=dict(color=color, dash="dash"))
    if label:
        fig.add_annotation(x=date, y=1, yref="paper", text=label, showarrow=False, yanchor="bottom",
                           font=dict(color=color, size=11))

def grey_lines(df, exclude=()):
    """All columns of df as ONE faint line trace (None breaks between machines), so hundreds of lines stay fast."""
    xs, ys = [], []
    for c in df.columns.drop(list(exclude), errors="ignore"):
        xs += list(df.index) + [None]
        ys += list(df[c]) + [None]
    return xs, ys

## 2. Data: one row per active machine per day

In [55]:
# Build one row per active machine per day, with features known IN ADVANCE only
daily  = load("1_prepared_views/prepared_daily_view.parquet")
events = load("4_context/events.parquet")
gm     = load("2_machine_decision/game_math.parquet")
cal    = load("2_machine_decision/calendar_day.parquet")

d = daily[daily["active"] & (daily["available_hours"] > 0)].copy()
d["date"] = pd.to_datetime(d["gaming_date"])
d = d.merge(cal[["gaming_date", "scheduled_seconds"]], on="gaming_date")
d["scheduled_hours"] = d["scheduled_seconds"] / 3600

# Event flag and event type from the events table (all announced well in advance)
d["event"] = 0
d["event_type"] = "none"
for _, e in events.iterrows():
    on = d["date"].between(pd.to_datetime(e["start_date"]), pd.to_datetime(e["end_date"]))
    d.loc[on, "event"] = 1
    d.loc[on, "event_type"] = e["event_type"]

# Event class (decidable in advance): short headline weekends vs long / weekday events
d["event_class"] = "none"
for _, e in events.iterrows():
    days = pd.date_range(e["start_date"], e["end_date"])
    cls = "headline weekend" if (len(days) <= 4 and any(x.dayofweek == 5 for x in days)) else "long / weekday"
    d.loc[d["date"].between(days[0], days[-1]), "event_class"] = cls

# Temperature (real NOAA data, Las Vegas airport). hot_deg = degrees above 30°C
weather = load("4_context/real_sources/weather_daily.parquet")
weather["date"] = pd.to_datetime(weather["source_date"])
for col in ["tmax_c", "tmin_c", "prcp_mm"]:
    weather[col] = weather[col].astype(float)
d = d.merge(weather[["date", "tmax_c", "tmin_c", "prcp_mm"]], on="date", how="left")
d["hot_deg"] = (d["tmax_c"] - 30).clip(lower=0)
d["rain"] = (d["prcp_mm"] > 0).astype(int)

# Zone foot traffic (coverage-adjusted, log). An OUTCOME known only after the day: used only in the traffic section
d["zone_traffic"] = np.log1p(d["zone_entries_observed"] * 24 / d["zone_counter_coverage_hours"])

# Game traits (lets a model generalise to a game it has never seen in a zone)
d = d.merge(gm[["config_id", "min_bet_cents", "max_bet_cents", "hit_probability"]], on="config_id", how="left")

# Target: coin-in per available hour (log), so machine counts and downtime don't distort demand
d["coin_per_hr"] = d["coin_in_usd"] / d["available_hours"]
d["y"] = np.log1p(d["coin_per_hr"])

train = d[d["date"] <  TRAIN_END].copy()
test  = d[d["date"] >= TRAIN_END].copy()

# Hours to convert a per-hour prediction back to dollars WITHOUT peeking at test-day downtime:
# planned hours = scheduled hours x the machine's availability in training (floor average if new)
avail_train = train.groupby("machine_id")["available_hours"].sum() / train.groupby("machine_id")["scheduled_hours"].sum()
floor_avail = train["available_hours"].sum() / train["scheduled_hours"].sum()
test["planned_hours"] = test["scheduled_hours"] * test["machine_id"].map(avail_train).fillna(floor_avail)

print(f"Temperature: training avg high {train['tmax_c'].mean():.1f}°C, {TEST_LABEL} avg high {test['tmax_c'].mean():.1f}°C")
print(f"Rows: train {len(train):,}  test {len(test):,}   machines: {d['machine_id'].nunique()}   floor availability in training: {floor_avail:.3%}")

Temperature: training avg high 22.9°C, June avg high 38.6°C
Rows: train 45,852  test 9,240   machines: 310   floor availability in training: 99.601%


C:\Users\stove\AppData\Local\Temp\ipykernel_19436\622093528.py:24: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  cls = "headline weekend" if (len(days) <= 4 and any(x.dayofweek == 5 for x in days)) else "long / weekday"
C:\Users\stove\AppData\Local\Temp\ipykernel_19436\622093528.py:24: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  cls = "headline weekend" if (len(days) <= 4 and any(x.dayofweek == 5 for x in days)) else "long / weekday"
C:\Users\stove\AppData\Local\Temp\ipykernel_19436\622093528.py:24: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  cls = "headline weekend" if (len(days) <= 4 and any(x.dayofweek == 5 for x in days)) else "long / weekday"
C:\Users\stove\AppData\Local\Temp\ipykernel_19436\622093528.py:24: Pandas4Warning: Timestamp.d

## 3. Evaluation: MAE at machine, game × zone, zone and floor level

In [56]:
# Evaluation: MAE at machine, game x zone, zone and FLOOR level (floor is comparable to model.ipynb's $48.4K)
def evaluate(pred_coin, name):
    t = test.assign(pred=pred_coin)
    out = {"model": name}
    out["machine-day MAE"] = (t["coin_in_usd"] - t["pred"]).abs().mean()
    for level, keys in [("game x zone", ["date", "zone_name", "theme"]), ("zone", ["date", "zone_name"]), ("floor", ["date"])]:
        g = t.groupby(keys)[["coin_in_usd", "pred"]].sum()
        out[f"{level} MAE"] = (g["coin_in_usd"] - g["pred"]).abs().mean()
    floor = t.groupby("date")[["coin_in_usd", "pred"]].sum()
    out["floor bias"] = floor["pred"].sum() / floor["coin_in_usd"].sum() - 1
    return out

def fmt_table(tbl):
    """Format a results table as $ and % (no jinja2 needed)."""
    out = tbl.copy()
    for c in out.columns:
        if "MAE" in c:
            out[c] = out[c].map(lambda v: f"${v:,.0f}")
        elif c == "floor bias":
            out[c] = out[c].map(lambda v: f"{v:+.1%}")
        elif c == "seconds":
            out[c] = out[c].map(lambda v: "" if pd.isna(v) else f"{v:.1f}")
    return out

results = []

## 4. Baselines
**A:** per zone × game × weekday median. **B:** the floor model from `model.ipynb` (one event boost), split across machines by training share. **C:** the same floor model with **two event classes** (headline weekend vs long / weekday).

In [57]:
# Baselines
# A) Per-cell weekday median of coin-in per hour (zone x game x weekday), with fallbacks, x planned hours
cell = train.groupby(["zone_name", "theme", "local_weekday"])["coin_per_hr"].median()
gw   = train.groupby(["theme", "local_weekday"])["coin_per_hr"].median()
wd   = train.groupby("local_weekday")["coin_per_hr"].median()
rate = pd.Series([cell.get((z, g, w), gw.get((g, w), wd[w])) for z, g, w in
                  zip(test["zone_name"], test["theme"], test["local_weekday"])], index=test.index)
pred_A = rate * test["planned_hours"]
results.append(evaluate(pred_A, "Baseline A: cell weekday median"))

# B) The floor model from model.ipynb (weekday median x event boost), split across machines by training share
fl = train.groupby(["date", "local_weekday", "event"])["coin_in_usd"].sum().reset_index()
wk = fl[fl["event"] == 0].groupby("local_weekday")["coin_in_usd"].median()
boost = (fl.loc[fl["event"] == 1, "coin_in_usd"] / fl.loc[fl["event"] == 1, "local_weekday"].map(wk)).median()
share = train.groupby("machine_id")["coin_in_usd"].sum() / train["coin_in_usd"].sum()
floor_pred = test["local_weekday"].map(wk) * np.where(test["event"] == 1, boost, 1)
n_per_day = test.groupby("date")["machine_id"].transform("count")
m_share = test["machine_id"].map(share).fillna(1 / n_per_day)
m_share = m_share / test.assign(s=m_share).groupby("date")["s"].transform("sum")     # shares sum to 1 each day
pred_B = floor_pred * m_share
results.append(evaluate(pred_B, "Baseline B: floor model split by share"))

# C) Floor model with TWO event classes (headline weekend vs long / weekday), split by share
flc = train.groupby(["date", "local_weekday", "event_class"])["coin_in_usd"].sum().reset_index()
wk_c = flc[flc["event_class"] == "none"].groupby("local_weekday")["coin_in_usd"].median()
boost_c = {c: (flc.loc[flc["event_class"] == c, "coin_in_usd"] / flc.loc[flc["event_class"] == c, "local_weekday"].map(wk_c)).median()
           for c in ["headline weekend", "long / weekday"]}
floor_pred_c = test["local_weekday"].map(wk_c) * test["event_class"].map(boost_c).fillna(1)
pred_C = floor_pred_c * m_share
results.append(evaluate(pred_C, "Baseline C: floor model + event classes"))
print("Event boosts learned from training months:", {"single": round(boost, 3)} | {k: round(v, 3) for k, v in boost_c.items()})

base_tbl = pd.DataFrame(results).set_index("model")
display(fmt_table(base_tbl))

# ---------- Visualization ----------
# 1) Floor total per day: actual vs the three baselines
fd = test.assign(A=pred_A, B=pred_B, C=pred_C).groupby("date")[["coin_in_usd", "A", "B", "C"]].sum()
fig = go.Figure()
fig.add_scatter(x=fd.index, y=fd["coin_in_usd"], name="Actual", line=dict(color=WHITE, width=3))
for col, name, color in [("A", "Baseline A: cell weekday median", ORANGE),
                         ("B", "Baseline B: floor model split by share", CYAN),
                         ("C", "Baseline C: floor model + event classes", GREEN)]:
    fig.add_scatter(x=fd.index, y=fd[col], name=name, line=dict(color=color, dash="dash", width=2))
fig.update_layout(title=f"Floor coin-in per day, {TEST_LABEL} holdout: baselines", yaxis_tickprefix="$",
                  hovermode="x unified", legend=dict(orientation="h", y=-0.2))
save(fig, "baselines_floor", height=430)

# 2) MAE at each level, as % of the average actual at that level (so levels are comparable)
levels = {"machine-day MAE": ["date", "machine_id"], "game x zone MAE": ["date", "zone_name", "theme"],
          "zone MAE": ["date", "zone_name"], "floor MAE": ["date"]}
pct = {}
for col, keys in levels.items():
    avg_actual = test.groupby(keys)["coin_in_usd"].sum().mean()
    pct[col.replace(" MAE", "")] = base_tbl[col] / avg_actual * 100
pct = pd.DataFrame(pct)
fig = go.Figure([go.Bar(x=list(pct.columns), y=pct.loc[name], name=name.split(":")[0], marker_color=c,
                        hovertemplate="%{x}: %{y:.1f}% of average actual<extra></extra>")
                 for name, c in zip(pct.index, [ORANGE, CYAN, GREEN])])
fig.update_layout(barmode="group", title="Error at each level (lower is better)",
                  yaxis_title="MAE as % of average actual", yaxis_ticksuffix="%", legend=dict(orientation="h", y=1.1))
save(fig, "baselines_error_by_level", height=400)

# 3) Daily floor error by weekday: where does each baseline miss?
err = fd.assign(A_err=(fd["A"] - fd["coin_in_usd"]) / fd["coin_in_usd"] * 100,
                B_err=(fd["B"] - fd["coin_in_usd"]) / fd["coin_in_usd"] * 100,
                C_err=(fd["C"] - fd["coin_in_usd"]) / fd["coin_in_usd"] * 100,
                wd=fd.index.strftime("%a"))
order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
e = err.groupby("wd")[["A_err", "B_err", "C_err"]].mean().reindex(order)
fig = go.Figure([go.Bar(x=order, y=e[k], name=lab, marker_color=c, hovertemplate="%{x}: %{y:+.1f}%<extra></extra>")
                 for k, lab, c in [("A_err", "Baseline A", ORANGE), ("B_err", "Baseline B", CYAN), ("C_err", "Baseline C", GREEN)]])
fig.add_hline(y=0, line_color=WHITE, line_width=1)
fig.update_layout(barmode="group", title="Floor error by weekday (+ = over-forecast)",
                  yaxis_title="Average floor error", yaxis_ticksuffix="%", legend=dict(orientation="h", y=1.1))
save(fig, "baselines_error_by_weekday", height=400)

Event boosts learned from training months: {'single': np.float64(1.108), 'headline weekend': np.float64(1.186), 'long / weekday': np.float64(1.023)}


,machine-day MAE,game x zone MAE,zone MAE,floor MAE,floor bias
model,,,,,
Baseline A: cell weekday median,$902,"$2,571","$12,230","$76,779",+4.5%
Baseline B: floor model split by share,$999,"$3,458","$13,882","$60,702",+4.6%
Baseline C: floor model + event classes,$979,"$3,233","$12,526","$46,339",+2.5%


zoom slider: yes


saved baselines_floor.html
zoom slider: no


saved baselines_error_by_level.html
zoom slider: no


saved baselines_error_by_weekday.html


**Analysis: baselines (June test)**
- June is a harder test than May–June: every baseline over-forecasts it (+2.5% to +4.6%) because June was the hottest, softest month.
- **Baseline C (event classes) is still the best floor baseline: $46.3K**, vs $60.7K for one event boost. With training through May, the single boost absorbs EDC and jumps to ×1.108, which then over-forecasts Locals Week. Event classes avoid that.
- Boosts learned January–May: headline weekend ×1.186, long/weekday ×1.023 (almost the same as with January–April, so the rule is stable).

## 5. ML models × feature sets
Add your own trials to the two dictionaries, then rerun this cell and everything below it.

In [58]:
# ML models x feature sets. Add your own trials to these two dicts.
feature_sets = {
    "day+game+zone+event":             (["local_weekday", "theme", "zone_name"], ["event"]),
    "day+game+zone+event type":        (["local_weekday", "theme", "zone_name", "event_type"], []),
    "day+game+zone+EVENT CLASS":       (["local_weekday", "theme", "zone_name", "event_class"], []),
    "day+game+BANK+EVENT CLASS":       (["local_weekday", "theme", "bank_id", "event_class"], []),
    "day+game+zone+EVENT CLASS+TEMP (oracle)":  (["local_weekday", "theme", "zone_name", "event_class"], ["tmax_c", "hot_deg", "rain"]),
    "day+game+BANK+EVENT CLASS+TEMP (oracle)":  (["local_weekday", "theme", "bank_id", "event_class"], ["tmax_c", "hot_deg", "rain"]),
    "day+game+BANK+event":             (["local_weekday", "theme", "bank_id"], ["event"]),
    "day+game TRAITS+zone+event":      (["local_weekday", "volatility_class", "zone_name"],
                                        ["event", "configured_rtp", "min_bet_cents", "max_bet_cents", "hit_probability"]),
}
models = {
    "Ridge":             lambda: Ridge(alpha=1.0),
    "Random forest":     lambda: RandomForestRegressor(n_estimators=150, min_samples_leaf=20, n_jobs=-1, random_state=0),
    "Gradient boosting": lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0),
}

predictions = {}

def run_trial(model_name, set_name):
    cats, nums = feature_sets[set_name]
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats),
                              ("num", StandardScaler(), nums)])
    pipe = make_pipeline(prep, models[model_name]())
    t0 = time.time()
    pipe.fit(train[cats + nums], train["y"])
    # Back-transform from log: this gives a median-like prediction, which is robust to event-day spikes
    # (a "smearing" correction to the mean was tested and over-predicted the floor by about 6%)
    pred_coin = np.expm1(pipe.predict(test[cats + nums])) * test["planned_hours"]
    predictions[f"{model_name} | {set_name}"] = pred_coin
    r = evaluate(pred_coin, f"{model_name} | {set_name}")
    r["seconds"] = time.time() - t0
    return r

for set_name in feature_sets:
    for model_name in models:
        results.append(run_trial(model_name, set_name))
        print(f"done: {model_name:18} | {set_name}")

table = pd.DataFrame(results).set_index("model").sort_values("floor MAE")
pd.set_option("display.width", 250)
print(f"Reference: best floor model (Baseline C) floor MAE = ${results[[r['model'] for r in results].index('Baseline C: floor model + event classes')]['floor MAE']:,.0f}")
fmt_table(table)

done: Ridge              | day+game+zone+event
done: Random forest      | day+game+zone+event
done: Gradient boosting  | day+game+zone+event
done: Ridge              | day+game+zone+event type
done: Random forest      | day+game+zone+event type
done: Gradient boosting  | day+game+zone+event type
done: Ridge              | day+game+zone+EVENT CLASS
done: Random forest      | day+game+zone+EVENT CLASS
done: Gradient boosting  | day+game+zone+EVENT CLASS
done: Ridge              | day+game+BANK+EVENT CLASS
done: Random forest      | day+game+BANK+EVENT CLASS
done: Gradient boosting  | day+game+BANK+EVENT CLASS
done: Ridge              | day+game+zone+EVENT CLASS+TEMP (oracle)
done: Random forest      | day+game+zone+EVENT CLASS+TEMP (oracle)
done: Gradient boosting  | day+game+zone+EVENT CLASS+TEMP (oracle)
done: Ridge              | day+game+BANK+EVENT CLASS+TEMP (oracle)
done: Random forest      | day+game+BANK+EVENT CLASS+TEMP (oracle)
done: Gradient boosting  | day+game+BANK+EVENT CLA

,machine-day MAE,game x zone MAE,zone MAE,floor MAE,floor bias,seconds
model,,,,,,
Gradient boosting | day+game+BANK+EVENT CLASS+TEMP (oracle),$854,"$2,300","$9,067","$41,446",-1.1%,2.1
Ridge | day+game+BANK+EVENT CLASS+TEMP (oracle),$872,"$2,530","$11,538","$41,700",-0.9%,0.2
Gradient boosting | day+game+zone+EVENT CLASS+TEMP (oracle),$864,"$2,269","$9,050","$41,883",-1.0%,1.2
Ridge | day+game+zone+EVENT CLASS+TEMP (oracle),$886,"$2,551","$11,593","$41,916",-1.0%,0.2
Gradient boosting | day+game+BANK+EVENT CLASS,$862,"$2,323","$9,822","$42,094",+1.1%,1.4
Random forest | day+game+BANK+EVENT CLASS,$878,"$2,402","$10,127","$42,516",+1.3%,2.8
Ridge | day+game+zone+EVENT CLASS,$893,"$2,588","$12,045","$42,585",+1.1%,0.1
Gradient boosting | day+game+zone+EVENT CLASS,$878,"$2,345","$9,969","$42,594",+1.0%,1.1
Random forest | day+game+zone+EVENT CLASS+TEMP (oracle),$879,"$2,440","$9,625","$42,711",-0.3%,1.2


**Analysis: ML trials (June test)**
- **Best usable model: gradient boosting with bank + game + weekday + event class, $42.1K floor MAE**, beating the best floor baseline ($46.3K) by 9%. Bottom-up ML now clearly wins on June.
- Gradient boosting is also the most accurate at the machine, game × zone and zone levels.
- **Temperature (oracle)** adds only a little ($42.1K → $41.4K) and flips the bias from +1.1% to −1.1%. Even with May's warm days in training, it can't learn June's heat effect well.
- Plain event yes/no models over-forecast June badly ($55–56K), confirming that event classes matter.

## 6. Reconciled: the best floor model (Baseline C) sets each day's total, ML splits it across machines

In [59]:
# Reconcile: rescale each day's machine forecasts so they add up to the BEST floor model's daily forecast (Baseline C).
# The floor model is best at the TOTAL; the ML model is best at SPLITTING it across games and zones.
floor_by_day = test.assign(fp=floor_pred_c).groupby("date")["fp"].first()      # best floor model (event classes)
for name in table.index[:3]:
    if name not in predictions:
        continue
    raw = predictions[name]
    scale = floor_by_day / test.assign(p=raw).groupby("date")["p"].sum()
    results.append(evaluate(raw * test["date"].map(scale), f"{name} + reconciled"))

table = pd.DataFrame(results).set_index("model").sort_values(["floor MAE", "game x zone MAE"])
print(f"Reference: best floor model (Baseline C) floor MAE = ${results[[r['model'] for r in results].index('Baseline C: floor model + event classes')]['floor MAE']:,.0f}")
fmt_table(table)

Reference: best floor model (Baseline C) floor MAE = $46,339


,machine-day MAE,game x zone MAE,zone MAE,floor MAE,floor bias,seconds
model,,,,,,
Gradient boosting | day+game+BANK+EVENT CLASS+TEMP (oracle),$854,"$2,300","$9,067","$41,446",-1.1%,2.1
Ridge | day+game+BANK+EVENT CLASS+TEMP (oracle),$872,"$2,530","$11,538","$41,700",-0.9%,0.2
Gradient boosting | day+game+zone+EVENT CLASS+TEMP (oracle),$864,"$2,269","$9,050","$41,883",-1.0%,1.2
Ridge | day+game+zone+EVENT CLASS+TEMP (oracle),$886,"$2,551","$11,593","$41,916",-1.0%,0.2
Gradient boosting | day+game+BANK+EVENT CLASS,$862,"$2,323","$9,822","$42,094",+1.1%,1.4
Random forest | day+game+BANK+EVENT CLASS,$878,"$2,402","$10,127","$42,516",+1.3%,2.8
Ridge | day+game+zone+EVENT CLASS,$893,"$2,588","$12,045","$42,585",+1.1%,0.1
Gradient boosting | day+game+zone+EVENT CLASS,$878,"$2,345","$9,969","$42,594",+1.0%,1.1
Random forest | day+game+zone+EVENT CLASS+TEMP (oracle),$879,"$2,440","$9,625","$42,711",-0.3%,1.2


**Analysis: reconciled**
On June, the bottom-up ML total is *better* than the floor model's ($42.1K vs $46.3K), so rescaling to the floor total makes the floor *worse*. With this split, use the ML sum directly for the total.

## 7. Totals: actual vs predicted, June

In [60]:
# Totals: actual vs predicted coin-in over the test period, by month and overall (best models only)
best_ml = [m for m in table.index if m in predictions][:3]
preds = {"Baseline A": pred_A, "Baseline B": pred_B, "Baseline C": pred_C} | {m: predictions[m] for m in best_ml}
t = test.assign(month=test["date"].dt.strftime("%m-%b"))
totals = pd.DataFrame({"Actual": t.groupby("month")["coin_in_usd"].sum()})
for name, p in preds.items():
    totals[name] = t.assign(p=p).groupby("month")["p"].sum()
totals.loc[f"{TEST_LABEL} total"] = totals.sum()
pct_diff = totals.drop(columns="Actual").div(totals["Actual"], axis=0) - 1
show = totals.map(lambda v: f"${v/1e6:,.2f}M")
for c in pct_diff.columns:
    show[c] = show[c] + pct_diff[c].map(lambda v: f"  ({v:+.1%})")
show.T

month,06-Jun,June total
Actual,$39.73M,$39.73M
Baseline A,$41.50M (+4.5%),$41.50M (+4.5%)
Baseline B,$41.54M (+4.6%),$41.54M (+4.6%)
Baseline C,$40.73M (+2.5%),$40.73M (+2.5%)
Gradient boosting | day+game+BANK+EVENT CLASS+TEMP (oracle),$39.31M (-1.1%),$39.31M (-1.1%)
Ridge | day+game+BANK+EVENT CLASS+TEMP (oracle),$39.38M (-0.9%),$39.38M (-0.9%)
Gradient boosting | day+game+zone+EVENT CLASS+TEMP (oracle),$39.35M (-1.0%),$39.35M (-1.0%)


**Analysis: totals**
The baselines over-forecast June by 2.5–4.6%. The best ML models land within about ±1% of June's actual $39.7M.

## 8. Model comparison (interactive)

In [61]:
# Model comparison dashboard (interactive): error at every level, the accuracy trade-off, and daily fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

res = pd.DataFrame(results).drop_duplicates("model").set_index("model")
levels = {"machine-day": ["date", "machine_id"], "game x zone": ["date", "zone_name", "theme"],
          "zone": ["date", "zone_name"], "floor": ["date"]}
avg_actual = {lvl: test.groupby(k)["coin_in_usd"].sum().mean() for lvl, k in levels.items()}
pct = pd.DataFrame({lvl: res[f"{lvl} MAE"] / avg_actual[lvl] * 100 for lvl in levels})
pct = pct.sort_values("floor")
short = [m.replace("Gradient boosting", "GBM").replace("Random forest", "RF").replace("Baseline ", "Base ")
           .replace(": cell weekday median", " (cell median)").replace(": floor model split by share", " (floor model)").replace(": floor model + event classes", " (floor + classes)")
         for m in pct.index]

family = lambda m: ("Baseline" if m.startswith("Baseline") else m.split(" |")[0]) + (" + reconciled" if "reconciled" in m else "")
colors = {"Baseline": GREY, "Ridge": CYAN, "Random forest": ORANGE, "Gradient boosting": GREEN,
          "Gradient boosting + reconciled": "#d62728", "Random forest + reconciled": "#9467bd", "Ridge + reconciled": "#8c564b"}

fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.18,
                    subplot_titles=("MAE as % of average actual (colour ranks models within each level)",
                                    "Splitting accuracy vs floor accuracy (best = bottom-left)"))

# 1) Heatmap: every model x every level
rank = (pct - pct.min()) / (pct.max() - pct.min())          # colour = rank WITHIN each level (0 = best model)
fig.add_trace(go.Heatmap(z=rank.values, x=list(pct.columns), y=short, colorscale="RdYlGn_r", zmin=0, zmax=1,
                         text=[[f"{v:.1f}%" for v in row] for row in pct.values], texttemplate="%{text}",
                         customdata=pct.values, hovertemplate="%{y}<br>%{x}: %{customdata:.2f}% of average actual<extra></extra>",
                         colorbar=dict(title="within level", tickvals=[0, 1], ticktext=["best", "worst"], x=0.47, len=0.9)),
              row=1, col=1)

# 2) Scatter: game x zone error (splitting) vs floor error (total)
for fam in dict.fromkeys(family(m) for m in res.index):
    sub = res[[family(m) == fam for m in res.index]]
    fig.add_trace(go.Scatter(x=sub["floor MAE"] / 1000, y=sub["game x zone MAE"], mode="markers",
                             name=fam, marker=dict(size=13, color=colors.get(fam, GREY), line=dict(width=1, color="white")),
                             text=sub.index, hovertemplate="%{text}<br>floor MAE $%{x:.1f}K<br>game x zone MAE $%{y:,.0f}<extra></extra>"),
                  row=1, col=2)
fig.add_vline(x=res.loc["Baseline C: floor model + event classes", "floor MAE"] / 1000, line_dash="dash", line_color="red", row=1, col=2,
              annotation_text="best floor model", annotation_position="bottom right")
fig.update_xaxes(title_text="Floor MAE ($K per day)", row=1, col=2)
fig.update_yaxes(title_text="Game x zone MAE ($ per day)", row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_layout(height=170 + 32 * len(pct), margin=dict(t=110), title=f"Bottom-up ML trials vs baselines, {TEST_LABEL} holdout", legend=dict(x=1.02, y=1))
save(fig, "model_comparison", height=170 + 32 * len(pct))

# 3) Daily floor total: actual vs the best floor baseline and our best USABLE model
#    (the "TEMP (oracle)" models use actual June weather, which isn't known in advance, so they're left out here)
actual = test.groupby("date")["coin_in_usd"].sum()
fp_line = test.assign(fp=floor_pred_c).groupby("date")["fp"].first()
best_usable = "Gradient boosting | day+game+BANK+EVENT CLASS"
fig2 = go.Figure()
fig2.add_scatter(x=actual.index, y=actual, name="Actual", line=dict(color=WHITE, width=3))
fig2.add_scatter(x=fp_line.index, y=fp_line, line=dict(color=GREY, dash="dash", width=2),
                 name=f"Baseline: floor model + event classes (MAE ${(fp_line - actual).abs().mean() / 1000:,.1f}K)")
if best_usable in predictions:
    s_ = test.assign(p=predictions[best_usable]).groupby("date")["p"].sum()
    fig2.add_scatter(x=s_.index, y=s_, line=dict(color=CYAN, dash="dot", width=2.5),
                     name=f"Our model: gradient boosting (MAE ${(s_ - actual).abs().mean() / 1000:,.1f}K)")
for _, e in events.iterrows():
    if pd.to_datetime(e["end_date"]) >= pd.Timestamp(TRAIN_END):
        fig2.add_vrect(x0=max(pd.to_datetime(e["start_date"]), pd.Timestamp(TRAIN_END)),
                       x1=pd.to_datetime(e["end_date"]) + pd.Timedelta(days=1),
                       fillcolor=GOLD, opacity=0.12, line_width=0, annotation_position="top left",
                       annotation_text=e["name"].replace("Gilded Ace ", ""), annotation_font_size=11)
fig2.update_layout(yaxis_title="Coin-in per day", yaxis_tickprefix="$", hovermode="x unified", margin=dict(t=60),
                   legend=dict(orientation="h", x=0, y=1.02, yanchor="bottom"))
save(fig2, "forecast_vs_actual", height=480)

zoom slider: no


saved model_comparison.html
zoom slider: yes


saved forecast_vs_actual.html


## 9. Next-day model with lag features and temperature
Lags (yesterday, last week, 7-day average, yesterday's change) and tomorrow's **weather forecast** are known one day ahead, so this model is for daily operations, **not** for the July plan. (We use actual temperature as a stand-in for a good next-day forecast.)

In [62]:
# Next-day model: add lag features (known only for a ONE-day-ahead forecast; NOT allowed for planning July)
d = d.sort_values(["machine_id", "date"])
g = d.groupby("machine_id")["y"]
d["lag1"] = g.shift(1)                                    # yesterday (log coin-in per hour)
d["lag7"] = g.shift(7)                                    # same weekday last week
d["chg_prev_day"] = d["lag1"] - g.shift(2)                # yesterday's relative change vs the day before
d["roll7"] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())   # average of the last 7 days

lag_cols = ["lag1", "lag7", "chg_prev_day", "roll7"]
train_nd = d[d["date"] <  TRAIN_END].dropna(subset=lag_cols).copy()
test_nd  = d[d["date"] >= TRAIN_END].dropna(subset=lag_cols).copy()
avail_nd = train_nd.groupby("machine_id")["available_hours"].sum() / train_nd.groupby("machine_id")["scheduled_hours"].sum()
test_nd["planned_hours"] = test_nd["scheduled_hours"] * test_nd["machine_id"].map(avail_nd).fillna(floor_avail)

def evaluate_on(t_df, pred_coin, name):
    t = t_df.assign(pred=pred_coin)
    out = {"model": name, "machine-day MAE": (t["coin_in_usd"] - t["pred"]).abs().mean()}
    for level, keys in [("game x zone", ["date", "zone_name", "theme"]), ("zone", ["date", "zone_name"]), ("floor", ["date"])]:
        s = t.groupby(keys)[["coin_in_usd", "pred"]].sum()
        out[f"{level} MAE"] = (s["coin_in_usd"] - s["pred"]).abs().mean()
    return out

next_day_sets = {
    "month-ahead features only":         (["local_weekday", "theme", "zone_name", "event_class"], []),
    "+ yesterday's change only":         (["local_weekday", "theme", "zone_name", "event_class"], ["chg_prev_day"]),
    "+ yesterday, last week, 7-day avg": (["local_weekday", "theme", "zone_name", "event_class"], ["lag1", "lag7", "roll7"]),
    "+ all lags and yesterday's change": (["local_weekday", "theme", "zone_name", "event_class"], lag_cols),
    "+ 3 lags + temperature (forecast)": (["local_weekday", "theme", "zone_name", "event_class"], ["lag1", "lag7", "roll7", "tmax_c", "hot_deg", "rain"]),
    "+ temperature only (forecast)":     (["local_weekday", "theme", "zone_name", "event_class"], ["tmax_c", "hot_deg", "rain"]),
}
nd_results = []
for name, (cats, nums) in next_day_sets.items():
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats),
                              ("num", StandardScaler(), nums)])
    pipe = make_pipeline(prep, HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0))
    pipe.fit(train_nd[cats + nums], train_nd["y"])
    pred = np.expm1(pipe.predict(test_nd[cats + nums])) * test_nd["planned_hours"]
    nd_results.append(evaluate_on(test_nd, pred, f"Gradient boosting | {name}"))

print(f"corr(today's change, yesterday's change): {(d['y'] - d['lag1']).corr(d['chg_prev_day']):.2f}  (negative = changes tend to reverse)")
fmt_table(pd.DataFrame(nd_results).set_index("model"))

corr(today's change, yesterday's change): -0.33  (negative = changes tend to reverse)


,machine-day MAE,game x zone MAE,zone MAE,floor MAE
model,,,,
Gradient boosting | month-ahead features only,$877,"$2,345","$9,925","$42,696"
Gradient boosting | + yesterday's change only,$874,"$2,327","$9,755","$42,440"
"Gradient boosting | + yesterday, last week, 7-day avg",$865,"$2,210","$8,776","$41,718"
Gradient boosting | + all lags and yesterday's change,$865,"$2,176","$8,442","$38,346"
Gradient boosting | + 3 lags + temperature (forecast),$865,"$2,165","$8,268","$39,924"
Gradient boosting | + temperature only (forecast),$866,"$2,269","$9,100","$42,723"


**Analysis: next-day model**
Using all lags (yesterday, last week, 7-day average, yesterday's change) gives the best next-day forecast for June, **$38.3K**. Adding temperature to the lags doesn't help. Valid one day ahead only.

## 10. What if we knew foot traffic? (oracle — NOT usable for forecasting)
Same-day zone entries are an **outcome**: they're only known after the day, so they can't be a forecast input (leakage). This section shows how much of the trough-day misses traffic *would* explain, to diagnose them, not to forecast.
- **June test:** gradient boosting (bank + game + weekday + event class) with and without same-day zone traffic, trained January–May.
- **May 6:** the same comparison with a model trained January–April and tested on May, since May is inside this notebook's training period.

In [63]:
# Oracle experiment: add same-day zone traffic (coverage-adjusted) and see what it fixes. NOT a usable forecast input.
def gbm_fit_predict(tr_df, te_df, cats, nums):
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats),
                              ("num", "passthrough", nums)])
    pipe = make_pipeline(prep, HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0))
    pipe.fit(tr_df[cats + nums], tr_df["y"])
    av = tr_df.groupby("machine_id")["available_hours"].sum() / tr_df.groupby("machine_id")["scheduled_hours"].sum()
    hrs = te_df["scheduled_hours"] * te_df["machine_id"].map(av).fillna(av.mean())
    return np.expm1(pipe.predict(te_df[cats + nums])) * hrs

base_cats = ["local_weekday", "theme", "bank_id", "event_class"]
runs = {"no traffic": [], "with same-day traffic (oracle)": ["zone_traffic"]}

def daily_compare(tr_df, te_df):
    out = te_df.groupby("date")[["coin_in_usd"]].sum()
    for name, nums in runs.items():
        out[name] = te_df.assign(p=gbm_fit_predict(tr_df, te_df, base_cats, nums)).groupby("date")["p"].sum()
    return out

june_cmp = daily_compare(train, test)
may_cmp  = daily_compare(d[d["date"] < "2025-05-01"], d[(d["date"] >= "2025-05-01") & (d["date"] < "2025-06-01")])

for label, cmp_ in [(f"{TEST_LABEL} (trained Jan–May)", june_cmp), ("May (trained Jan–Apr)", may_cmp)]:
    maes = {k: (cmp_[k] - cmp_["coin_in_usd"]).abs().mean() for k in runs}
    print(f"{label:28}  floor MAE  no traffic ${maes['no traffic']:,.0f}   with traffic ${maes['with same-day traffic (oracle)']:,.0f}")

trough_days = ["2025-05-05", "2025-05-06", "2025-05-07", "2025-06-10", "2025-06-11", "2025-06-16", "2025-06-17", "2025-06-18", "2025-06-19"]
both = pd.concat([may_cmp, june_cmp])
tbl_t = both.loc[pd.to_datetime(trough_days)].copy()
for k in runs:
    tbl_t[f"{k} error"] = tbl_t[k] / tbl_t["coin_in_usd"] - 1
tbl_t.index = tbl_t.index.strftime("%a %b %d")
print("\nTrough days: forecast error with and without same-day traffic")
display(tbl_t[[f"{k} error" for k in runs]].map(lambda v: f"{v:+.1%}"))

# One continuous timeline, May 1 – June 30 (single x-axis, so it gets a zoom slider)
tl = pd.concat([may_cmp, june_cmp]).sort_index()
fig = go.Figure()
fig.add_scatter(x=tl.index, y=tl["coin_in_usd"], name="Actual", line=dict(color=WHITE, width=3))
fig.add_scatter(x=tl.index, y=tl["no traffic"], name="Forecast without traffic (usable)",
                line=dict(color=CYAN, dash="dash", width=2))
fig.add_scatter(x=tl.index, y=tl["with same-day traffic (oracle)"], name="With same-day traffic (oracle, not usable)",
                line=dict(color=RED, dash="dot", width=2))
fig.add_vrect(x0=pd.Timestamp("2025-05-05"), x1=pd.Timestamp("2025-05-08"), fillcolor="lightgray", opacity=0.12, line_width=0,
              annotation_text="quiet", annotation_position="top left", annotation_font_size=11)
fig.add_vrect(x0=pd.Timestamp("2025-06-09"), x1=pd.Timestamp("2025-06-16"), fillcolor=GOLD, opacity=0.15, line_width=0,
              annotation_text="Locals Week", annotation_position="top left", annotation_font_size=11)
fig.add_vrect(x0=pd.Timestamp("2025-06-16"), x1=pd.Timestamp("2025-06-20"), fillcolor="lightgray", opacity=0.12, line_width=0,
              annotation_text="lull + heat", annotation_position="top left", annotation_font_size=11)
date_line(fig, "2025-06-01", "← May: trained Jan–Apr  |  June: trained Jan–May →", color=GREY)
fig.update_layout(yaxis_title="Floor coin-in per day", yaxis_tickprefix="$", hovermode="x unified",
                  margin=dict(t=70), legend=dict(orientation="h", x=0, y=1.08, yanchor="bottom"))
save(fig, "traffic_oracle", height=480)

June (trained Jan–May)        floor MAE  no traffic $42,094   with traffic $26,556
May (trained Jan–Apr)         floor MAE  no traffic $39,400   with traffic $29,080

Trough days: forecast error with and without same-day traffic


,no traffic error,with same-day traffic (oracle) error
date,,
Mon May 05,+12.1%,+8.1%
Tue May 06,+9.8%,+4.8%
Wed May 07,+2.2%,-0.3%
Tue Jun 10,-6.1%,-5.6%
Wed Jun 11,-7.4%,-6.7%
Mon Jun 16,+4.4%,+0.9%
Tue Jun 17,+6.7%,+4.2%
Wed Jun 18,+4.5%,+2.4%
Thu Jun 19,+7.1%,+5.0%


zoom slider: yes


saved traffic_oracle.html


**Analysis: what traffic would fix (oracle)**
- Knowing same-day traffic would cut floor error by **about 37% in June** ($42.1K → $26.6K) and **26% in May** ($39.4K → $29.1K). Most daily misses are about **how many people came in**.
- **May 5–6 (quiet stretch):** errors shrink from +12% / +10% to +8% / +5%. Fewer people came, and traffic explains about half the miss.
- **Week after Locals Week (Jun 16–19):** errors shrink from +4–7% to +1–5%. The lull was mostly lower turnout.
- **Locals Week weekdays (Jun 10–11):** traffic barely helps (−6/−7% stays about the same). The extra play came from **people spending more per visit**, not just more visitors, which fits a promotion aimed at locals.
- **Why we can't use it:** traffic is only known after the day. It's useful for diagnosing misses and measuring actions (e.g. Back Forty entries after the signs), not as a forecast input.

## 11. Heat and demand (explanatory)
Looking back with actual temperatures is allowed: it explains what happened, it does not forecast.

In [64]:
# Does heat explain the June softness? Floor-level check (explanatory, looking back)
import statsmodels.formula.api as smf
fl_w = d.groupby("date").agg(coin=("coin_in_usd", "sum"), wd=("local_weekday", "first"), ec=("event_class", "first"),
                             tmax=("tmax_c", "first"), hot=("hot_deg", "first"), rain=("rain", "first")).reset_index()
fl_w["log_coin"] = np.log(fl_w["coin"])
m_heat = smf.ols("log_coin ~ C(wd) + C(ec) + hot + rain", data=fl_w).fit()
eff = np.exp(m_heat.params["hot"]) - 1
lo, hi = np.exp(m_heat.conf_int().loc["hot"]) - 1
print(f"All six months: each °C above 30°C changes daily coin-in by {eff:+.2%} (95% CI {lo:+.2%} to {hi:+.2%})")
m_tr = smf.ols("log_coin ~ C(wd) + C(ec) + hot + rain", data=fl_w[fl_w["date"] < TRAIN_END]).fit()
print(f"Training months only: {np.exp(m_tr.params['hot']) - 1:+.2%} per °C  (learned without June)")
print(f"Implied effect at June's average high ({fl_w.loc[fl_w['date'].dt.month == 6, 'tmax'].mean():.1f}°C): "
      f"{(1 + eff) ** fl_w.loc[fl_w['date'].dt.month == 6, 'hot'].mean() - 1:+.1%}")

resid = m_heat.resid + m_heat.params["hot"] * fl_w["hot"]
xs = np.linspace(fl_w["tmax"].min(), fl_w["tmax"].max(), 100)
fig = go.Figure()
fig.add_scatter(x=fl_w["tmax"], y=(np.exp(resid) - 1) * 100, mode="markers", name="Day",
                marker=dict(size=7, color=fl_w["date"].dt.month, colorscale="RdBu_r", showscale=True,
                            colorbar=dict(title="Month")),
                text=fl_w["date"].dt.strftime("%b %d"),
                hovertemplate="%{text}: %{x:.1f}°C, %{y:+.1f}% vs normal<extra></extra>")
fig.add_scatter(x=xs, y=((1 + eff) ** np.clip(xs - 30, 0, None) - 1) * 100, mode="lines", name="Fitted heat effect",
                line=dict(color=GOLD, width=3))
fig.add_hline(y=0, line_color=FAINT, line_width=1)
fig.update_layout(title="Hotter days run below normal", xaxis_title="Daily high temperature (°C, NOAA, Las Vegas airport)",
                  yaxis_title="Floor coin-in vs normal for weekday & event", yaxis_ticksuffix="%",
                  legend=dict(orientation="h", y=1.1))
save(fig, "heat_effect", height=420)

All six months: each °C above 30°C changes daily coin-in by -0.24% (95% CI -0.38% to -0.09%)
Training months only: +0.08% per °C  (learned without June)
Implied effect at June's average high (38.6°C): -2.0%
zoom slider: no


saved heat_effect.html


**Analysis: heat**
- Over all six months, each °C above 30°C lowers coin-in by 0.24% (about −2% at June's 38.6°C average high, roughly half of June's shortfall).
- **Even with May in training, the heat effect comes out near zero (+0.08% per °C)**: May wasn't hot enough. The effect only shows up once June is included, so it's a July adjustment based on January–June, not something validated on a holdout.

## 12. July 2025 forecast
Validation is done, so every model is **refit on all six months** (January–June) before forecasting July.
- **Floor total:** weekday median × event-class boost; the Jul 18–20 promotion is a *headline weekend*.
- **Uncertainty:** daily P10/P90 = day-to-day noise × month-level drift, measured on January–June.
- **Heat:** effect per °C above 30°C estimated on January–June, applied to July's **normal** high (about 41°C, NOAA 1991–2020 normals; an assumption to confirm and cite). It's a forecastable input because normals are known in advance.
- **Split by zone:** gradient boosting (bank + game + weekday + event class), refit on January–June, predicts each machine on the floor on June 30, reconciled so each day adds up to the floor P50.
- **Expected win (theo)** = forecast coin-in × each machine's designed hold (1 − RTP).

In [65]:
# July 2025 forecast: floor total (P10/P50/P90) + heat adjustment + split by zone (bottom-up ML, reconciled)
import statsmodels.formula.api as smf

JULY_NORMAL_HIGH_C = 41.0      # assumption: NOAA 1991–2020 normal July high, Las Vegas airport (~106°F) — confirm & cite
PROMO = ("2025-07-18", "2025-07-20")

# ---------- 1) Floor total: weekday median x event-class boost, refit on ALL six months ----------
fl6 = d.groupby("date").agg(coin=("coin_in_usd", "sum"), wd=("local_weekday", "first"),
                            ec=("event_class", "first"), hot=("hot_deg", "first"), rain=("rain", "first")).reset_index()
wk6 = fl6[fl6["ec"] == "none"].groupby("wd")["coin"].median()
boost6 = {c: (fl6.loc[fl6["ec"] == c, "coin"] / fl6.loc[fl6["ec"] == c, "wd"].map(wk6)).median()
          for c in ["headline weekend", "long / weekday"]}

# P10/P90: day-to-day noise x month-level drift, measured on Jan–Jun
fl6["p50"] = fl6["wd"].map(wk6) * fl6["ec"].map(boost6).fillna(1)
fl6["r"] = fl6["coin"] / fl6["p50"]
fl6["m"] = fl6["date"].dt.month
daily_noise = fl6["r"] / fl6.groupby("m")["r"].transform("median")
month_level = fl6.groupby("m")["r"].median() / fl6["r"].median()
q10, q90 = np.quantile([a * b for a in daily_noise for b in month_level], [0.10, 0.90])

# Heat effect learned from Jan–Jun (log coin-in per °C above 30°C), applied to July's normal high
m_heat = smf.ols("np.log(coin) ~ C(wd) + C(ec) + hot + rain", data=fl6).fit()
heat_per_deg = m_heat.params["hot"]
heat_factor = np.exp(heat_per_deg * max(0, JULY_NORMAL_HIGH_C - 30))

jul = pd.DataFrame({"date": pd.date_range("2025-07-01", "2025-07-30")})
jul["wd"] = jul["date"].dt.strftime("%a")
jul["ec"] = np.where(jul["date"].between(*PROMO), "headline weekend", "none")
jul["p50_no_promo"] = jul["wd"].map(wk6)
jul["p50"] = jul["p50_no_promo"] * jul["ec"].map(boost6).fillna(1)
jul["p50_heat"] = jul["p50"] * heat_factor
jul["p10"], jul["p90"] = jul["p50_heat"] * q10, jul["p50_heat"] * q90
jul["success_line"] = jul["p50_no_promo"] * heat_factor * q90          # promo must beat the no-promo P90

# ---------- 2) Split by machine and zone: gradient boosting refit on all six months, reconciled ----------
cats_j, nums_j = ["local_weekday", "theme", "bank_id", "event_class"], []
prep_j = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats_j)])
gbm_j = make_pipeline(prep_j, HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0))
gbm_j.fit(d[cats_j], d["y"])

last_day = d["date"].max()
floor_now = d[d["date"] == last_day][["machine_id", "zone_name", "bank_id", "theme", "configured_rtp"]]   # machines on the floor Jun 30
avail6 = d.groupby("machine_id")["available_hours"].sum() / d.groupby("machine_id")["scheduled_hours"].sum()
grid = floor_now.merge(jul[["date", "wd", "ec"]], how="cross").rename(columns={"wd": "local_weekday", "ec": "event_class"})
grid["planned_hours"] = 24 * grid["machine_id"].map(avail6).fillna(avail6.mean())
grid["raw"] = np.expm1(gbm_j.predict(grid[cats_j])) * grid["planned_hours"]
scale = jul.set_index("date")["p50_heat"] / grid.groupby("date")["raw"].sum()
grid["coin_pred"] = grid["raw"] * grid["date"].map(scale)                      # reconciled to the floor total
grid["theo_pred"] = grid["coin_pred"] * (1 - grid["configured_rtp"])           # expected win = coin-in x designed hold

# ---------- 3) Summary ----------
june = d[d["date"].dt.month == 6]
z = pd.DataFrame({"June actual coin-in": june.groupby("zone_name")["coin_in_usd"].sum(),
                  "July forecast coin-in": grid.groupby("zone_name")["coin_pred"].sum(),
                  "July forecast theo": grid.groupby("zone_name")["theo_pred"].sum()})
z["change vs June"] = (z["July forecast coin-in"] / 30) / (z["June actual coin-in"] / 30) - 1
z.loc["FLOOR"] = z.sum(numeric_only=True)
z.loc["FLOOR", "change vs June"] = (z.loc["FLOOR", "July forecast coin-in"] / 30) / (z.loc["FLOOR", "June actual coin-in"] / 30) - 1

print(f"Event boosts (Jan–Jun): headline weekend x{boost6['headline weekend']:.3f}, long/weekday x{boost6['long / weekday']:.3f}")
print(f"Heat: {np.exp(heat_per_deg) - 1:+.2%} per °C above 30°C (Jan–Jun)  ->  July at {JULY_NORMAL_HIGH_C:.0f}°C: x{heat_factor:.3f}")
print(f"Daily band: P10 {q10-1:+.1%}, P90 {q90-1:+.1%}")
print(f"\nJuly total coin-in  P50 ${jul['p50_heat'].sum()/1e6:,.1f}M   (P10 ${jul['p10'].sum()/1e6:,.1f}M – P90 ${jul['p90'].sum()/1e6:,.1f}M)"
      f"   | without heat ${jul['p50'].sum()/1e6:,.1f}M | without promo ${(jul['p50_no_promo']*heat_factor).sum()/1e6:,.1f}M")
print(f"July expected win (theo): ${grid['theo_pred'].sum()/1e6:,.2f}M")
promo = jul[jul["ec"] == "headline weekend"]
print(f"Promotion Jul 18–20: P50 ${promo['p50_heat'].sum()/1e6:,.2f}M; counts as working if actual > ${promo['success_line'].sum()/1e6:,.2f}M (no-promo P90)\n")
z_show = z.copy()
for c in z_show.columns:
    z_show[c] = z_show[c].map(lambda v: f"{v:+.1%}") if "change" in c else z_show[c].map(lambda v: f"${v:,.0f}")
display(z_show)

# ---------- 4) Chart ----------
hist = d.groupby("date")["coin_in_usd"].sum()
fig = make_subplots(rows=2, cols=1, row_heights=[0.6, 0.4], vertical_spacing=0.12,
                    subplot_titles=("Floor coin-in: actual (mid-April to June) and July forecast — pan left for Jan–Mar",
                                    "July forecast by zone (gradient boosting, reconciled to the floor P50)"))
fig.add_trace(go.Scatter(x=hist.index, y=hist, name="Actual Jan–Jun", line=dict(color=WHITE, width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=list(jul["date"]) + list(jul["date"][::-1]), y=list(jul["p90"]) + list(jul["p10"][::-1]),
                         fill="toself", fillcolor="rgba(44,160,44,0.18)", line=dict(width=0), name="July P10–P90"), row=1, col=1)
fig.add_trace(go.Scatter(x=jul["date"], y=jul["p50_heat"], name="July P50 (with heat + promo)",
                         line=dict(color="#2ca02c", width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=jul["date"], y=jul["p50"], name="July P50 without heat adjustment",
                         line=dict(color="#2ca02c", width=1, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=jul["date"], y=jul["success_line"], name="No-promo P90 (promotion success line)",
                         line=dict(color="gray", width=1.5, dash="dash")), row=1, col=1)
fig.add_vrect(x0=pd.Timestamp(PROMO[0]), x1=pd.Timestamp(PROMO[1]) + pd.Timedelta(days=1), fillcolor="gold", opacity=0.3,
              line_width=0, annotation_text="Promotion", annotation_position="top left", row=1, col=1)
fig.add_vline(x=pd.Timestamp("2025-07-04"), line_dash="dashdot", line_color=WHITE, opacity=0.5, row=1, col=1)
fig.add_annotation(x=pd.Timestamp("2025-07-04"), y=hist.min(), text="July 4: extra uncertainty", showarrow=False,
                   xanchor="left", yanchor="bottom", font=dict(size=10), row=1, col=1)
fig.add_vline(x=pd.Timestamp("2025-07-01"), line_color="gray", opacity=0.4, row=1, col=1)

zone_order = grid.groupby("zone_name")["coin_pred"].sum().sort_values().index
zcols = {"Back Forty": "#d62728"}
palette = ["#1f77b4", "#ff7f0e", "#9467bd", "#8c564b", "#e377c2", "#17becf", "#bcbd22"]
for i, zn in enumerate(zone_order):
    s = grid[grid["zone_name"] == zn].groupby("date")["coin_pred"].sum()
    fig.add_trace(go.Bar(x=s.index, y=s, name=zn, marker_color=zcols.get(zn, palette[i % len(palette)]),
                         legend="legend2"), row=2, col=1)
# Two legends: line legend beside the top chart, zone legend beside the bottom chart
fig.update_layout(legend=dict(x=1.02, y=1.0, yanchor="top"),
                  legend2=dict(x=1.02, y=0.36, yanchor="top", title_text="Zones"))
fig.update_layout(barmode="stack", height=900, hovermode="x unified",
                  title="July 2025 forecast (refit on Jan–Jun): floor total with uncertainty, heat and promotion, split by zone")
fig.update_xaxes(range=[pd.Timestamp("2025-04-14"), pd.Timestamp("2025-07-31")], row=1, col=1)   # zoom; pan left to see Jan–Mar
fig.update_yaxes(title_text="Coin-in per day ($)", row=1, col=1)
fig.update_yaxes(title_text="Coin-in per day ($)", row=2, col=1)
save(fig, "july_forecast", height=900)

Event boosts (Jan–Jun): headline weekend x1.191, long/weekday x1.034
Heat: -0.24% per °C above 30°C (Jan–Jun)  ->  July at 41°C: x0.974
Daily band: P10 -4.5%, P90 +4.1%

July total coin-in  P50 $39.9M   (P10 $38.0M – P90 $41.5M)   | without heat $40.9M | without promo $39.0M
July expected win (theo): $3.77M
Promotion Jul 18–20: P50 $5.49M; counts as working if actual > $4.80M (no-promo P90)



,June actual coin-in,July forecast coin-in,July forecast theo,change vs June
zone_name,,,,
Back Forty,"$4,059,400","$3,884,629","$371,806",-4.3%
Bar Row,"$6,204,112","$6,381,740","$596,487",+2.9%
Center Pit,"$12,095,480","$12,052,976","$1,143,826",-0.4%
Garage Entrance,"$4,364,173","$4,467,856","$421,884",+2.4%
High-Limit Room,"$2,478,116","$2,435,088","$238,797",-1.7%
Hotel Corridor,"$4,722,452","$4,825,703","$450,037",+2.2%
Main Entrance,"$5,806,818","$5,802,392","$542,305",-0.1%
FLOOR,"$39,730,550","$39,850,384","$3,765,141",+0.3%


zoom slider: no


saved july_forecast.html


## 13. Findings (train January–May, test June)
- **Best usable forecast for June: gradient boosting, bank + game + weekday + event class ($42.1K floor MAE)**, 9% better than the best floor baseline ($46.3K) and about 31% better than one event boost ($60.7K).
- **Event classes are essential.** With a single boost, EDC in training inflates the boost and Locals Week gets over-forecast. Headline weekend ×1.19 vs long/weekday ×1.02 stays stable whichever months are used for training.
- **Temperature:** explains part of June's softness looking back (−0.24% per °C above 30°C, about −2% in June), but can't be learned before June. Use it as a July adjustment from January–June data with normal temperatures, not as a validated feature.
- **Foot traffic (oracle):** would cut June error by about 37%. Most misses are turnout, except Locals Week, where locals spent more per visit. It's an outcome, so it's for diagnosis and measuring actions, not forecasting.
- **Next-day forecasts** with lags reach $38.3K; daily operations only.
- **July forecast** (refit on January–June): P50 ≈ $39.9M coin-in (P10 $38.0M – P90 $41.5M), ≈ $3.8M expected win; the Jul 18–20 promotion counts as working if its coin-in beats ≈ $4.8M.
- **Comparing splits:** the May–June test (`ml_trials.ipynb`) and the June test agree on the main points: event classes matter most, gradient boosting splits best, and temperature and traffic help only as oracles.